# Day 7 Lab 06: Silver Cleaning and Standardization

        **Layer:** Silver  
        **Python reference:** `Lab_Files/labs/lab_06_silver_cleaning_and_standardization.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Apply the shared `cleaned_orders` transformation.
- Standardize strings, parse timestamps, and cast numeric fields.
- Write the clean-candidate Silver table.

**Dependency:** Run Lab/Notebook 04 first so Bronze orders are available.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from day7_common import LAKE_DIR, OUTPUT_DIR, cleaned_orders, ensure_output_dirs, read_bronze_orders, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Read Bronze

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook06SilverCleaning")

bronze = read_bronze_orders(spark)
bronze.select("event_id", "order_id", "status", "amount", "currency", "event_time").show(10, truncate=False)

## 3. Apply Cleaning Transformation

In [ ]:
clean_candidate = cleaned_orders(bronze)

## 4. Inspect Cleaned Types and Values

In [ ]:
clean_candidate.printSchema()
preview = clean_candidate.select(
    "event_id", "order_id", "customer_id", "product_id", "status", "amount", "currency", "event_time_ts"
).orderBy("event_id")
preview.show(20, truncate=False)

## 5. Write Silver Clean Candidate

In [ ]:
silver_path = LAKE_DIR / "silver" / "orders_clean_candidate"
clean_candidate.write.mode("overwrite").parquet(str(silver_path))
write_csv_dir(preview, OUTPUT_DIR / "notebook_06_clean_candidate_preview")
print(f"Clean candidate rows: {clean_candidate.count()}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()